In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display
from pathlib import Path
import plotly.graph_objects as go

# 분석 기간 & 원본 컬럼 설정
DATA_PATH = Path('../data')
START_YEAR, END_YEAR = 2022, 2025

# 데이터 분석
def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding = 'euc-kr')


# 그래프 글꼴
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 변수 설정 및 원본데이터 호출
BERTH_COL = '선석'
ARRIVAL_COL = '접안(예정)일시'
DEPARTURE_COL = '출항(예정)일시'
HANDLING_COLS = ['양하', '적하', 'Shift']
YEAR_COL, MONTH_COL = '기준년도', '월'
PORT_MONTH_COLS = [f'신항{i}부두' for i in range(1, 6)]

# 2. 원본 데이터 불러오기
port1 = load_dataset('port1')
port2 = load_dataset('port2')
port3 = load_dataset('port3')
port4 = load_dataset('port4')
port5 = load_dataset('port5')
port_month = load_dataset('port_month')
record = load_dataset("record")

ports = [
    (1, port1),
    (2, port2),
    (3, port3),
    (4, port4),
    (5, port5)
]

# ======================================
# 2. 분석 대상 설정
# ======================================

# 모든 선석 포함
expected_berths = {
    1: [1, 2, 3],
    2: [4, 5, 6, 7, 8, 9],  # 9선석 포함
    3: [1, 2, 3, 4],
    4: [1, 2, 3, 4],        # 4선석 포함
    5: [1, 2, 3, 4]
}

# 분석 기간: 2022~2025년
START = pd.Timestamp("2022-01-01")
END = pd.Timestamp("2026-01-01")

# 분석 기간 총 시간
period_hours = (END - START).total_seconds() / 3600


# 부두 선석별 월 대기시간 및 재항시간

In [1]:

# port1~5 합치기
port_list = []

for i, p in ports:
    p = p.copy()

    # port3만 '선박명'이라는 열 이름을 사용
    p = p.rename(columns={
        "선박명": "선명",
        "선석": "접안자료선석"
    })

    p["부두"] = i
    port_list.append(p)

port = pd.concat(port_list, ignore_index=True)
port = port.drop_duplicates().copy()
port["접안ID"] = range(len(port))



# 날짜 변환
port["접안시각"] = pd.to_datetime(
    port["접안(예정)일시"], format="mixed", errors="coerce"
)
port["선석종료시각"] = pd.to_datetime(
    port["출항(예정)일시"], format="mixed", errors="coerce"
)

record["입항시각"] = pd.to_datetime(
    record["입항일시"], format="mixed", errors="coerce"
)
record["출항시각"] = pd.to_datetime(
    record["출항일시"], format="mixed", errors="coerce"
)


# record의 계선장소에서 부두·선석 추출
record[["부두", "선석"]] = (
    record["계선장소"]
    .str.extract(r"신항\s*([1-5])\s*부두\s*(\d+)\s*선석")
    .apply(pd.to_numeric)
)


# 선명 공백·대소문자 통일
for df in [port, record]:
    df["연결선명"] = (
        df["선명"].str.upper()
        .str.replace(r"\s+", "", regex=True)
    )


# 필요한 정보가 없는 기록 제외
port = port.dropna(
    subset=["연결선명", "접안시각", "선석종료시각"]
).copy()

record = record.dropna(
    subset=["부두", "선석", "연결선명", "입항시각", "출항시각"]
).copy()

port = port[
    port["선석종료시각"] >= port["접안시각"]
].copy()

record = record[
    (record["출항시각"] >= record["입항시각"])
    & (record["입항시각"] >= "2022-01-01")
    & (record["입항시각"] < "2026-01-01")
].copy()

record[["부두", "선석"]] = record[["부두", "선석"]].astype(int)

record["재항시간"] = (
    record["출항시각"] - record["입항시각"]
).dt.total_seconds() / 3600

record["월"] = record["입항시각"].dt.to_period("M").astype(str)

ships = pd.merge_asof(
    record.sort_values("출항시각"),

    port[[
        "부두", "연결선명", "접안자료선석",
        "접안시각", "선석종료시각", "접안ID"
    ]].sort_values("선석종료시각"),

    left_on="출항시각",
    right_on="선석종료시각",
    by=["부두", "연결선명"],
    direction="nearest",
    tolerance=pd.Timedelta(hours=24)
)

ships["출항시각차이"] = (
    ships["출항시각"] - ships["선석종료시각"]
).abs()


# 입항 이후, 출항 이전에 접안한 기록만 사용
valid = ships[
    (ships["접안시각"] >= ships["입항시각"])
    & (ships["접안시각"] <= ships["출항시각"])
].copy()


# 같은 접안 기록을 여러 입항에 중복 사용하지 않음
valid = (
    valid.sort_values("출항시각차이")
    .drop_duplicates("접안ID")
    .copy()
)

valid["대기시간"] = (
    valid["접안시각"] - valid["입항시각"]
).dt.total_seconds() / 3600

group_cols = ["부두", "선석", "월"]

# record 전체 기준
monthly_all = (
    record.groupby(group_cols)
    .agg(
        전체입항건수=("재항시간", "size"),
        전체_평균재항시간=("재항시간", "mean")
    )
)

# 접안 기록까지 연결된 표본 기준
monthly_matched = (
    valid.groupby(group_cols)
    .agg(
        대기시간계산건수=("대기시간", "size"),
        평균대기시간=("대기시간", "mean"),
        연결표본_평균재항시간=("재항시간", "mean")
    )
)

monthly = monthly_all.join(monthly_matched)

monthly["대기시간계산건수"] = (
    monthly["대기시간계산건수"].fillna(0).astype(int)
)

monthly["분석포함률(%)"] = (
    monthly["대기시간계산건수"]
    / monthly["전체입항건수"] * 100
)

monthly = monthly.reset_index().sort_values(group_cols)

display(monthly.round(2))

,부두,선석,월,전체입항건수,전체_평균재항시간,대기시간계산건수,평균대기시간,연결표본_평균재항시간,분석포함률(%)
0,1,1,2022-01,26,24.80,23,0.93,24.85,88.46
1,1,1,2022-02,19,29.99,18,0.34,31.05,94.74
2,1,1,2022-03,24,21.32,24,0.28,21.32,100.00
3,1,1,2022-04,25,25.73,23,0.23,24.51,92.00
4,1,1,2022-05,22,27.97,21,0.35,26.34,95.45
...,...,...,...,...,...,...,...,...,...
908,5,4,2025-08,20,21.66,19,0.39,21.76,95.00
909,5,4,2025-09,23,23.06,23,3.79,23.06,100.00
910,5,4,2025-10,28,23.22,28,3.04,23.22,100.00
911,5,4,2025-11,23,23.78,23,2.29,23.78,100.00


# 체류생산성

In [2]:

# 3. 원본을 복사한 뒤 날짜, 물동량, 재항시간을 계산하고 이상값 제외
def prepare_port_data(data, port):
    data = data.copy()
    # 원본 데이터는 수정하지 않고 복사본에서 날짜를 변환
    data[ARRIVAL_COL] = pd.to_datetime(data[ARRIVAL_COL], errors='coerce')
    data[DEPARTURE_COL] = pd.to_datetime(data[DEPARTURE_COL], errors='coerce')
    # 양하 + 적하 + Shift를 계산하기 위해 물동량 컬럼을 숫자로 변환
    data[HANDLING_COLS] = data[HANDLING_COLS].apply(
        lambda column: pd.to_numeric(column.astype(str).str.replace(',', '', regex=False), errors='coerce')
    )
    # 선박별 처리 TEU = 양하 + 적하 + Shift
    data['processing_teu'] = data[HANDLING_COLS].sum(axis=1, min_count=len(HANDLING_COLS))
    # 선박별 재항시간 = 출항(예정)일시 - 접안(예정)일시
    data['stay_hours'] = (data[DEPARTURE_COL] - data[ARRIVAL_COL]).dt.total_seconds() / 3600
    data['year_month'] = data[ARRIVAL_COL].dt.to_period('M').astype(str)
    data['date'] = data[ARRIVAL_COL].dt.to_period('M').dt.to_timestamp()
    data['port'] = port
    data['berth'] = data[BERTH_COL].astype(str)
    # 분석 기간과 이상값 제외 기준을 적용
    valid = (
        data[ARRIVAL_COL].dt.year.between(START_YEAR, END_YEAR)
        & data[HANDLING_COLS].ge(0).all(axis=1)
        & data['processing_teu'].gt(0)
        & data['stay_hours'].gt(0)
        & ~data.duplicated()
    )
    return data.loc[valid].copy()

# 4. 부두별 복사본을 결합해 분석용 전체 데이터 생성
raw_port_data = {'port1': port1, 'port2': port2, 'port3': port3, 'port4': port4, 'port5': port5}
all_data = pd.concat(
    [prepare_port_data(data, port) for port, data in raw_port_data.items()],
    ignore_index=True,
)

# 5. 월별·부두별·선석별 물동량과 재항시간 합계 계산
berth_result = (
    all_data.groupby(['port', 'year_month', 'date', 'berth'], as_index=False)
    .agg(berth_volume=('processing_teu', 'sum'), stay_hours=('stay_hours', 'sum'))
)
# 같은 부두·월의 전체 물동량을 구해 선석별 배분 비율 계산
berth_result['port_volume'] = berth_result.groupby(['port', 'year_month'])['berth_volume'].transform('sum')
berth_result['berth_share'] = berth_result['berth_volume'] / berth_result['port_volume'].replace(0, np.nan)

# 6. 부두별 월간 천TEU 데이터를 선석별 배분에 사용할 수 있도록 정리
port_month_work = port_month.copy()
port_month_work[YEAR_COL] = pd.to_numeric(port_month_work[YEAR_COL], errors='coerce')
port_month_work[MONTH_COL] = pd.to_numeric(port_month_work[MONTH_COL], errors='coerce')
port_month_work = port_month_work[port_month_work[YEAR_COL].between(START_YEAR, END_YEAR)].copy()
port_month_work['year_month'] = (
    port_month_work[YEAR_COL].astype(int).astype(str)
    + '-' + port_month_work[MONTH_COL].astype(int).astype(str).str.zfill(2)
)
port_month_long = port_month_work.melt(
    id_vars='year_month', value_vars=PORT_MONTH_COLS,
    var_name='source_port', value_name='port_thousand_teu'
)
port_month_long['port'] = port_month_long['source_port'].map(
    {column: f'port{i}' for i, column in enumerate(PORT_MONTH_COLS, 1)}
)
port_month_long['port_thousand_teu'] = pd.to_numeric(port_month_long['port_thousand_teu'], errors='coerce')

# 7. 부두별 천TEU를 선석별 물동량 비율로 배분
berth_result = berth_result.merge(
    port_month_long[['port', 'year_month', 'port_thousand_teu']],
    on=['port', 'year_month'], how='left'
)
# 선석별 추정 천TEU = 부두별 월간 천TEU × 선석별 배분 비율
berth_result['estimated_thousand_teu'] = berth_result['port_thousand_teu'] * berth_result['berth_share']
# 선석별 체류생산성 = 선석별 추정 천TEU ÷ 선석별 재항시간
berth_result['stay_productivity'] = berth_result['estimated_thousand_teu'] / berth_result['stay_hours'].replace(0, np.nan)

# 8. 부두별 그래프를 위해 선석별 체류생산성의 평균 계산
port_result = (
    berth_result.groupby(['port', 'year_month'], as_index=False)
    .agg(stay_productivity=('stay_productivity', 'mean'), berth_count=('berth', 'nunique'))
)
port_result['date'] = pd.to_datetime(port_result['year_month'])

# 9. 월별·부두별 평균 체류생산성 Plotly 그래프 생성
fig = px.line(
    port_result, x='date', y='stay_productivity', color='port', markers=True,
    hover_data={'date': False, 'year_month': True, 'stay_productivity': ':.4f', 'berth_count': True}
)
fig.update_layout(
    title='월별·부두별 평균 체류생산성',
    xaxis_title='년월', yaxis_title='선석별 평균 체류생산성 (천TEU/시간)',
    hovermode='x unified', template='plotly_white', legend_title='부두'
)
fig.update_xaxes(dtick='M3', tickformat='%Y-%m', tickangle=-45)
fig.show()

# 10. 부두별 및 선석별 계산 결과 목록 출력
port_table = port_result.rename(columns={
    'port': '부두', 'year_month': '년월', 'berth_count': '선석수',
    'stay_productivity': '체류생산성 (천TEU/시간)'
})[['부두', '년월', '선석수', '체류생산성 (천TEU/시간)']]

berth_table = berth_result.rename(columns={
    'port': '부두', 'year_month': '년월', 'berth': '선석',
    'berth_share': '선석별배분비율', 'estimated_thousand_teu': '선석별천TEU',
    'stay_hours': '재항시간(시간)', 'stay_productivity': '체류생산성 (천TEU/시간)'
})[['부두', '년월', '선석', '선석별배분비율', '선석별천TEU', '재항시간(시간)', '체류생산성 (천TEU/시간)']]

print('부두별 결과 목록')
display(port_table)
print('선석별 결과 목록')
display(berth_table)

부두별 결과 목록


,부두,년월,선석수,체류생산성 (천TEU/시간)
0,port1,2022-01,3,0.126491
1,port1,2022-02,3,0.125776
2,port1,2022-03,3,0.131716
3,port1,2022-04,3,0.124049
4,port1,2022-05,3,0.126521
...,...,...,...,...
235,port5,2025-08,4,0.106197
236,port5,2025-09,4,0.111932
237,port5,2025-10,4,0.106545
238,port5,2025-11,4,0.107638


선석별 결과 목록


,부두,년월,선석,선석별배분비율,선석별천TEU,재항시간(시간),체류생산성 (천TEU/시간)
0,port1,2022-01,1,0.285384,70.775195,646.000000,0.109559
1,port1,2022-01,2,0.295936,73.392070,569.900000,0.128781
2,port1,2022-01,3,0.418680,103.832735,735.700000,0.141135
3,port1,2022-02,1,0.286692,65.078989,575.750000,0.113033
4,port1,2022-02,2,0.322298,73.161699,576.083333,0.126998
...,...,...,...,...,...,...,...
907,port5,2025-11,4,0.231328,48.347600,478.416667,0.101058
908,port5,2025-12,1,0.278396,56.236063,531.933333,0.105720
909,port5,2025-12,2,0.367253,74.185047,606.416667,0.122333
910,port5,2025-12,3,0.154572,31.223558,316.116667,0.098772


# 선석점유율

In [ ]:

# ======================================
# 2. 부두별 작업시간 계산
# ======================================

frames = []

for number, port in ports:

    if port is None:
        raise FileNotFoundError(
            f"{number}부두 CSV 파일을 확인하세요."
        )

    data = port.copy()
    data.columns = data.columns.str.strip()

    # 접안시각 변환
    data["접안시각"] = pd.to_datetime(
        data["접안(예정)일시"],
        errors="coerce"
    )

    # 출항시각 변환
    data["출항시각"] = pd.to_datetime(
        data["출항(예정)일시"],
        errors="coerce"
    )

    # 선석 번호 변환
    data["선석"] = pd.to_numeric(
        data["선석"],
        errors="coerce"
    )

    # 오류 및 누락 데이터 제외
    invalid = (
        data[["접안시각", "출항시각", "선석"]]
        .isna().any(axis=1)
        | (data["출항시각"] < data["접안시각"])
    )

    if invalid.any():
        print(f"{number}부두: 오류·누락 {invalid.sum()}건 제외")

    data = data.loc[~invalid].copy()

    # 해당 부두의 선석만 선택
    data = data[
        data["선석"].isin(expected_berths[number])
    ].copy()

    # 분석 기간과 겹치는 기록 선택
    data = data.loc[
        (data["접안시각"] < END)
        & (data["출항시각"] > START)
    ].copy()

    # 분석 기간에 포함되는 시간만 계산
    work_start = data["접안시각"].clip(lower=START)
    work_end = data["출항시각"].clip(upper=END)

    # 작업시간 = 출항시각 - 접안시각
    data["작업시간_시간"] = (
        work_end - work_start
    ).dt.total_seconds() / 3600

    data["부두"] = number

    frames.append(
        data[["부두", "작업시간_시간"]]
    )

detail = pd.concat(frames, ignore_index=True)


# ======================================
# 3. 부두별 평균 선석점유율 계산
# ======================================

# 부두별 총 작업시간 합산
result = (
    detail.groupby("부두")
    .agg(
        기록수=("작업시간_시간", "count"),
        총작업시간_시간=("작업시간_시간", "sum")
    )
    .reset_index()
)

# 모든 부두가 표시되도록 설정
result = (
    result.set_index("부두")
    .reindex([1, 2, 3, 4, 5])
    .reset_index()
)

# 선석 수
berth_counts = {
    number: len(berths)
    for number, berths in expected_berths.items()
}

result["선석수"] = result["부두"].map(berth_counts)

# 평균 선석점유율 계산
result["평균선석점유율_퍼센트"] = (
    result["총작업시간_시간"]
    / (result["선석수"] * period_hours)
    * 100
)

# 부두 이름
result["부두명"] = result["부두"].astype(str) + "부두"

print("\n[2022~2025년 부산 신항 부두별 평균 선석점유율]")

print(
    result[
        [
            "부두명",
            "선석수",
            "기록수",
            "총작업시간_시간",
            "평균선석점유율_퍼센트"
        ]
    ].round(2).to_string(index=False, na_rep="자료 없음")
)


# ======================================
# 4. Plotly 막대그래프
# ======================================

port_colors = {
    1: "#4C78A8",
    2: "#F58518",
    3: "#54A24B",
    4: "#E45756",
    5: "#B279A2"
}

fig = go.Figure()

for _, row in result.iterrows():

    number = int(row["부두"])
    value = row["평균선석점유율_퍼센트"]

    fig.add_trace(
        go.Bar(
            x=[f"{number}부두"],
            y=[value],
            name=f"{number}부두",
            marker_color=port_colors[number],

            text=(
                f"{value:.2f}%"
                if pd.notna(value)
                else "자료 없음"
            ),
            textposition="outside",

            customdata=[[
                row["총작업시간_시간"],
                row["선석수"],
                row["기록수"]
            ]],

            hovertemplate=(
                "부두: %{x}<br>"
                "평균 선석점유율: %{y:.2f}%<br>"
                "총 작업시간: %{customdata[0]:,.1f}시간<br>"
                "선석 수: %{customdata[1]:.0f}개<br>"
                "기록 수: %{customdata[2]:,.0f}건"
                "<extra></extra>"
            )
        )
    )

# ======================================
# 5. 그래프 디자인
# ======================================

maximum = result["평균선석점유율_퍼센트"].max()

upper = (
    max(100, maximum * 1.15)
    if pd.notna(maximum)
    else 100
)

fig.update_layout(

    title=dict(
        text="2022~2025년 부산 신항 부두별 평균 선석점유율",
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    ),

    xaxis=dict(
        title="부두",
        categoryorder="array",
        categoryarray=[
            "1부두",
            "2부두",
            "3부두",
            "4부두",
            "5부두"
        ],
        showgrid=False
    ),

    yaxis=dict(
        title="평균 선석점유율 (%)",
        range=[0, upper],
        ticksuffix="%",
        showgrid=True,
        gridcolor="#E5E7EB"
    ),

    barmode="group",
    template="plotly_white",

    width=1100,
    height=600,

    bargap=0.35,
    showlegend=False,

    margin=dict(
        t=100,
        b=70,
        l=80,
        r=40
    )
)

fig.show()

In [ ]:
# ======================================
# 2. 작업시간 계산
# ======================================

frames = []

for number, port in ports:

    data = port.copy()
    data.columns = data.columns.str.strip()

    data["접안시각"] = pd.to_datetime(
        data["접안(예정)일시"],
        errors="coerce"
    )

    data["출항시각"] = pd.to_datetime(
        data["출항(예정)일시"],
        errors="coerce"
    )

    data["선석"] = pd.to_numeric(
        data["선석"], errors="coerce"
    )

    invalid = (
        data[["접안시각", "출항시각", "선석"]].isna().any(axis=1)
        | (data["출항시각"] < data["접안시각"])
    )

    if invalid.any():
        print(f"{number}부두: 오류·누락 {invalid.sum()}건 제외")

    data = data.loc[~invalid].copy()

    data = data[
        data["선석"].isin(expected_berths[number])
    ].copy()

    data = data.loc[
        (data["접안시각"] < END)
        & (data["출항시각"] > START)
    ].copy()

    work_start = data["접안시각"].clip(lower=START)
    work_end = data["출항시각"].clip(upper=END)

    data["작업시간_시간"] = (
        work_end - work_start
    ).dt.total_seconds() / 3600

    data["부두"] = number

    frames.append(
        data[["부두", "선석", "작업시간_시간"]]
    )

detail = pd.concat(frames, ignore_index=True)

# ======================================
# 3. 선석점유율 계산
# ======================================

result = (
    detail.groupby(["부두", "선석"])
    .agg(총작업시간_시간=("작업시간_시간", "sum"))
)

expected_index = pd.MultiIndex.from_tuples(
    [
        (number, berth)
        for number, berths in expected_berths.items()
        for berth in berths
    ],
    names=["부두", "선석"]
)

result = result.reindex(expected_index).reset_index()

result["평균선석점유율_퍼센트"] = (
    result["총작업시간_시간"] / period_hours * 100
)

print("\n[2022~2025년 부두별·선석별 평균 선석점유율]")
print(result.round(2).to_string(index=False, na_rep="자료 없음"))

# ======================================
# 4. Plotly 바그래프
# ======================================

port_colors = {
    1: "#4C78A8",
    2: "#F58518",
    3: "#54A24B",
    4: "#E45756",
    5: "#B279A2"
}

fig = go.Figure()

for number in expected_berths:

    temp = result[result["부두"] == number].copy()

    labels = (
        temp["부두"].astype(str) + "부두 "
        + temp["선석"].astype(int).astype(str) + "선석"
    )

    values = temp["평균선석점유율_퍼센트"]

    fig.add_trace(
        go.Bar(
            x=labels,
            y=values,
            name=f"{number}부두",
            marker_color=port_colors[number],
            text=[
                f"{v:.2f}%" if pd.notna(v) else "자료 없음"
                for v in values
            ],
            textposition="outside",
            customdata=temp[["총작업시간_시간"]].to_numpy(),
            hovertemplate=(
                "선석: %{x}<br>"
                "평균 선석점유율: %{y:.2f}%<br>"
                "총 작업시간: %{customdata[0]:,.1f}시간"
                "<extra></extra>"
            )
        )
    )

# ======================================
# 5. 그래프 디자인
# ======================================

maximum = result["평균선석점유율_퍼센트"].max()

upper = (
    max(100, maximum * 1.15)
    if pd.notna(maximum)
    else 100
)

fig.update_layout(
    title=dict(
        text="2022~2025년 부두별·선석별 평균 선석점유율",
        x=0.5,
        font=dict(size=20)
    ),
    xaxis=dict(
        title="부두 / 선석",
        tickangle=-45
    ),
    yaxis=dict(
        title="평균 선석점유율 (%)",
        range=[0, upper],
        ticksuffix="%"
    ),
    barmode="group",
    template="plotly_white",
    width=1400,
    height=650,
    bargap=0.25,
    legend_title="부두"
)

# ======================================
# 6. PyCharm 노트북 내부에 출력
# ======================================

fig.show()

In [ ]:
# ======================================
# 3. 데이터 전처리
# ======================================

frames = []

for number, port in ports:

    data = port.copy()
    data.columns = data.columns.str.strip()

    # 접안시각 변환
    data["접안시각"] = pd.to_datetime(
        data["접안(예정)일시"],
        format="%Y-%m-%d %H:%M",
        errors="coerce"
    )

    # 출항시각 변환
    data["출항시각"] = pd.to_datetime(
        data["출항(예정)일시"],
        format="%Y-%m-%d %H:%M",
        errors="coerce"
    )

    # 선석 번호 변환
    data["선석"] = pd.to_numeric(
        data["선석"],
        errors="coerce"
    )

    # 오류 및 누락 데이터 제외
    invalid = (
        data[["접안시각", "출항시각", "선석"]]
        .isna().any(axis=1)
        | (data["출항시각"] <= data["접안시각"])
    )

    if invalid.any():
        print(f"{number}부두: 오류·누락 {invalid.sum()}건 제외")

    data = data.loc[~invalid].copy()

    # 해당 부두 선석만 선택
    data = data[
        data["선석"].isin(expected_berths[number])
    ].copy()

    # 분석 기간과 겹치는 기록만 선택
    data = data.loc[
        (data["접안시각"] < END)
        & (data["출항시각"] > START)
    ].copy()

    data["부두"] = number

    frames.append(
        data[[
            "부두",
            "선석",
            "접안시각",
            "출항시각"
        ]]
    )

detail = pd.concat(frames, ignore_index=True)

# ======================================
# 4. 월별·부두별·선석별 작업시간 계산
# ======================================

monthly_frames = []

months = pd.date_range(
    start=START,
    end=END,
    freq="MS",
    inclusive="left"
)

for month_start in months:

    month_end = month_start + pd.DateOffset(months=1)

    # 해당 월과 겹치는 기록 선택
    temp = detail.loc[
        (detail["접안시각"] < month_end)
        & (detail["출항시각"] > month_start)
    ].copy()

    if temp.empty:
        continue

    # 해당 월에 포함된 시간만 계산
    work_start = temp["접안시각"].clip(lower=month_start)
    work_end = temp["출항시각"].clip(upper=month_end)

    temp["작업시간_시간"] = (
        work_end - work_start
    ).dt.total_seconds() / 3600

    temp["기준월"] = month_start

    monthly_frames.append(
        temp[[
            "기준월",
            "부두",
            "선석",
            "작업시간_시간"
        ]]
    )

# ======================================
# 5. 월별 작업시간 합산
# ======================================

if monthly_frames:

    monthly_detail = pd.concat(
        monthly_frames,
        ignore_index=True
    )

    result = (
        monthly_detail.groupby(
            ["기준월", "부두", "선석"]
        )
        .agg(
            기록수=("작업시간_시간", "count"),
            총작업시간_시간=("작업시간_시간", "sum")
        )
    )

else:

    result = pd.DataFrame(
        columns=["기록수", "총작업시간_시간"]
    )

    result.index = pd.MultiIndex.from_arrays(
        [[], [], []],
        names=["기준월", "부두", "선석"]
    )


# ======================================
# 6. 모든 월·부두·선석 포함
# ======================================

expected_index = pd.MultiIndex.from_tuples(
    [
        (month, number, berth)
        for month in months
        for number, berths in expected_berths.items()
        for berth in berths
    ],
    names=["기준월", "부두", "선석"]
)

result = (
    result.reindex(expected_index)
    .reset_index()
)

# ======================================
# 7. 월별 선석점유율 계산
# ======================================

# 해당 월 총 시간
result["월총시간"] = (
    (
        result["기준월"] + pd.DateOffset(months=1)
    ) - result["기준월"]
).dt.total_seconds() / 3600

# 월별 선석점유율 계산
result["선석점유율_퍼센트"] = (
    result["총작업시간_시간"]
    / result["월총시간"]
    * 100
)

# 연도 및 월
result["연도"] = result["기준월"].dt.year
result["월"] = result["기준월"].dt.month

# 열 정리
result = result[
    [
        "연도",
        "월",
        "부두",
        "선석",
        "기록수",
        "총작업시간_시간",
        "월총시간",
        "선석점유율_퍼센트"
    ]
]

result = result.sort_values(
    ["연도", "월", "부두", "선석"]
).reset_index(drop=True)


# ======================================
# 8. 계산 결과 출력
# ======================================

print("\n[2022~2025년 월별·부두별·선석별 선석점유율]")

print(
    result.round(2).to_string(
        index=False,
        na_rep="자료 없음"
    )
)